# 03_PREPROCESSING

## Objectif

Ce notebook transforme `processed.app_train_enriched` et `processed.app_test_enriched` en jeux directement utilisables pour la modelisation.

Le preprocessing applique ici :

- impute les valeurs manquantes ;
- encode les variables categorielles selon leur nature et leur cardinalite ;
- reduit la skewness des variables numeriques trop asymetriques ;
- aligne parfaitement train et test ;
- exporte les jeux finaux en CSV et en base PostgreSQL.

Sorties produites :

- `df_app_train_model_ready`
- `df_app_test_model_ready`
- `processed.app_train_model_ready`
- `processed.app_test_model_ready`


In [ ]:
from pathlib import Path
from io import StringIO
import os
import re
import sys

import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
from sklearn.preprocessing import PowerTransformer

project_root = Path.cwd()
if not (project_root / "scripts").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.Fonctions_MODEL import encode_cat_col

load_dotenv(project_root / ".env")

conn = psycopg2.connect(
    host=os.getenv("PGHOST"),
    port=int(os.getenv("PGPORT", 5432)),
    user=os.getenv("PGUSER"),
    password=os.getenv("PGPASSWORD"),
    dbname=os.getenv("PGDATABASE", "home_credit"),
)

id_col = "SK_ID_CURR"
target_col = "TARGET"

print("Connexion PostgreSQL ouverte.")
print("Projet :", project_root)


Connexion PostgreSQL ouverte.
Projet : c:\Users\kevin\Documents\P6\P6_MLOps_1-2


In [ ]:
# Chargement des tables enrichies produites dans le notebook 02.
df_app_train_enriched = pd.read_sql("SELECT * FROM processed.app_train_enriched", conn)
df_app_test_enriched = pd.read_sql("SELECT * FROM processed.app_test_enriched", conn)

print("Train enrichi :", df_app_train_enriched.shape)
print("Test enrichi  :", df_app_test_enriched.shape)

assert id_col in df_app_train_enriched.columns
assert id_col in df_app_test_enriched.columns
assert target_col in df_app_train_enriched.columns
assert target_col not in df_app_test_enriched.columns

train_feature_cols = [c for c in df_app_train_enriched.columns if c != target_col]
test_feature_cols = df_app_test_enriched.columns.tolist()
print("Colonnes train hors target = colonnes test :", train_feature_cols == test_feature_cols)


C:\Users\kevin\AppData\Local\Temp\ipykernel_13228\2291735876.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_app_train_enriched = pd.read_sql("SELECT * FROM processed.app_train_enriched", conn)
C:\Users\kevin\AppData\Local\Temp\ipykernel_13228\2291735876.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_app_test_enriched = pd.read_sql("SELECT * FROM processed.app_test_enriched", conn)


Train enrichi : (251289, 111)
Test enrichi  : (48744, 110)
Colonnes train hors target = colonnes test : True


In [ ]:
# Separation train / test et identification des types de colonnes.
df_train = df_app_train_enriched.copy()
df_test = df_app_test_enriched.copy()

y_train = df_train[target_col].astype(int).copy()
X_train = df_train.drop(columns=[target_col]).copy()
X_test = df_test.copy()

cat_cols = X_train.select_dtypes(include=["object", "string", "category"]).columns.tolist()
num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
num_cols = [c for c in num_cols if c != id_col]

binary_num_cols = [c for c in num_cols if X_train[c].dropna().nunique() <= 2]
continuous_num_cols = [c for c in num_cols if c not in binary_num_cols]

# Variables ordinales definies par connaissance metier.
ordinal_mappings = {
    "NAME_EDUCATION_TYPE": [
        "Lower secondary",
        "Secondary / secondary special",
        "Incomplete higher",
        "Higher education",
        "Academic degree",
    ]
}

ordinal_cols = [c for c in ordinal_mappings if c in cat_cols]
nominal_cols = [c for c in cat_cols if c not in ordinal_cols]

low_cardinality_threshold = 10
rare_min_count = 100
skew_threshold = 1.0

low_card_nominal_cols = [c for c in nominal_cols if X_train[c].nunique(dropna=False) <= low_cardinality_threshold]
high_card_nominal_cols = [c for c in nominal_cols if X_train[c].nunique(dropna=False) > low_cardinality_threshold]

preprocessing_plan = pd.DataFrame(
    [{"variable": id_col, "type": "identifier", "strategy": "kept_for_traceability"}]
    + [{"variable": c, "type": "numeric_binary", "strategy": "median_impute_only"} for c in binary_num_cols]
    + [{"variable": c, "type": "numeric_continuous", "strategy": "median_impute_then_optional_skew_reduction"} for c in continuous_num_cols]
    + [{"variable": c, "type": "categorical_ordinal", "strategy": "fill_missing_then_manual_ordinal_mapping"} for c in ordinal_cols]
    + [{"variable": c, "type": "categorical_nominal_low_card", "strategy": "fill_missing_then_onehot"} for c in low_card_nominal_cols]
    + [{"variable": c, "type": "categorical_nominal_high_card", "strategy": "fill_missing_then_rare_grouping_then_binary_encoding"} for c in high_card_nominal_cols]
).sort_values(["type", "variable"]).reset_index(drop=True)

print("Nb colonnes numeriques :", len(num_cols))
print("Nb colonnes categorielles :", len(cat_cols))
print("Ordinales :", ordinal_cols)
print("Nominales faible cardinalite :", low_card_nominal_cols)
print("Nominales forte cardinalite  :", high_card_nominal_cols)
display(preprocessing_plan)


Nb colonnes numeriques : 97
Nb colonnes categorielles : 12
Ordinales : ['NAME_EDUCATION_TYPE']
Nominales faible cardinalite : ['CODE_GENDER', 'EMERGENCYSTATE_MODE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'NAME_CONTRACT_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_INCOME_TYPE', 'WALLSMATERIAL_MODE']
Nominales forte cardinalite  : ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']


,variable,type,strategy
0,OCCUPATION_TYPE,categorical_nominal_high_card,fill_missing_then_rare_grouping_then_binary_en...
1,ORGANIZATION_TYPE,categorical_nominal_high_card,fill_missing_then_rare_grouping_then_binary_en...
2,CODE_GENDER,categorical_nominal_low_card,fill_missing_then_onehot
3,EMERGENCYSTATE_MODE,categorical_nominal_low_card,fill_missing_then_onehot
4,FONDKAPREMONT_MODE,categorical_nominal_low_card,fill_missing_then_onehot
...,...,...,...
105,prev_avg_amt_credit,numeric_continuous,median_impute_then_optional_skew_reduction
106,prev_avg_rate_down_payment,numeric_continuous,median_impute_then_optional_skew_reduction
107,prev_nb_approved,numeric_continuous,median_impute_then_optional_skew_reduction
108,prev_nb_records,numeric_continuous,median_impute_then_optional_skew_reduction


In [ ]:
# Imputation robuste des manquants.
X_train_num = X_train[num_cols].copy()
X_test_num = X_test[num_cols].copy()
numeric_medians = X_train_num.median()
X_train_num = X_train_num.fillna(numeric_medians)
X_test_num = X_test_num.fillna(numeric_medians)

# Les variables continues sont castees en float avant les transformations
# pour eviter les erreurs de type lors de l'affectation de valeurs transforme es.
if continuous_num_cols:
    X_train_num = X_train_num.astype({col: float for col in continuous_num_cols})
    X_test_num = X_test_num.astype({col: float for col in continuous_num_cols})

X_train_cat = X_train[cat_cols].copy().fillna("__MISSING__")
X_test_cat = X_test[cat_cols].copy().fillna("__MISSING__")

# Reduction de la skewness numerique.
skew_before = X_train_num[continuous_num_cols].skew().sort_values(ascending=False)
skewed_cols = skew_before[skew_before.abs() > skew_threshold].index.tolist()

# log1p uniquement si la variable est non negative dans le train ET dans le test.
non_negative_skewed_cols = [
    c for c in skewed_cols
    if X_train_num[c].min() >= 0 and X_test_num[c].min() >= 0
]
signed_skewed_cols = [c for c in skewed_cols if c not in non_negative_skewed_cols]

if non_negative_skewed_cols:
    X_train_num.loc[:, non_negative_skewed_cols] = np.log1p(X_train_num[non_negative_skewed_cols].astype(float))
    X_test_num.loc[:, non_negative_skewed_cols] = np.log1p(X_test_num[non_negative_skewed_cols].astype(float))

yeo_johnson_transformer = None
if signed_skewed_cols:
    yeo_johnson_transformer = PowerTransformer(method="yeo-johnson", standardize=False)
    X_train_num.loc[:, signed_skewed_cols] = yeo_johnson_transformer.fit_transform(
        X_train_num[signed_skewed_cols].astype(float)
    )
    X_test_num.loc[:, signed_skewed_cols] = yeo_johnson_transformer.transform(
        X_test_num[signed_skewed_cols].astype(float)
    )

print("Nb variables skewees :", len(skewed_cols))
print("Transformation log1p :", len(non_negative_skewed_cols))
print("Transformation Yeo-Johnson :", len(signed_skewed_cols))
display(skew_before.head(20))


c:\Users\kevin\Documents\P6\P6_MLOps_1-2\.venv\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


TypeError: Invalid value '[ -5884.29300885   -727.68389456 -19175.15097948 ...   -772.57697374
  18007.01800829 -33952.61219203]' for dtype 'int64'

In [ ]:
# Encodage des variables categorielles en reutilisant la fonction du repo
# pour les strategies one-hot et binary.
def fit_transform_cat_column(train_df, test_df, col_name, encoding_type):
    train_encoded, encoder = encode_cat_col(
        train_df[[col_name]].copy(),
        col_name,
        encoding_type,
    )

    if encoding_type == "onehot":
        test_array = encoder.transform(test_df[[col_name]])
        test_cols = encoder.get_feature_names_out([col_name])
        test_encoded = pd.DataFrame(test_array, columns=test_cols, index=test_df.index)
    elif encoding_type == "binary":
        test_encoded = encoder.transform(test_df[[col_name]])
        test_encoded.index = test_df.index
    else:
        raise ValueError("Type d'encodage inconnu pour cette fonction.")

    return train_encoded, test_encoded, encoder


# Variables ordinales : mapping manuel pour controler explicitement l'ordre
# et gerer proprement les valeurs inattendues ou manquantes en -1.
X_train_ord = pd.DataFrame(index=X_train.index)
X_test_ord = pd.DataFrame(index=X_test.index)
for col in ordinal_cols:
    mapping = {category: idx for idx, category in enumerate(ordinal_mappings[col])}
    X_train_ord[col] = X_train_cat[col].map(mapping).fillna(-1).astype(int)
    X_test_ord[col] = X_test_cat[col].map(mapping).fillna(-1).astype(int)

# Groupement des modalites rares pour les nominales a forte cardinalite.
def group_rare_modalities(train_series, test_series, min_count=100, rare_label="__RARE__"):
    counts = train_series.value_counts(dropna=False)
    kept = counts[counts >= min_count].index.tolist()
    return train_series.where(train_series.isin(kept), rare_label), test_series.where(test_series.isin(kept), rare_label)

X_train_nominal = X_train_cat[nominal_cols].copy() if nominal_cols else pd.DataFrame(index=X_train.index)
X_test_nominal = X_test_cat[nominal_cols].copy() if nominal_cols else pd.DataFrame(index=X_test.index)
for col in high_card_nominal_cols:
    X_train_nominal[col], X_test_nominal[col] = group_rare_modalities(
        X_train_nominal[col], X_test_nominal[col], min_count=rare_min_count
    )

# One-hot pour les nominales de faible cardinalite.
ohe_train_frames = []
ohe_test_frames = []
for col in low_card_nominal_cols:
    train_ohe_col, test_ohe_col, _ = fit_transform_cat_column(X_train_nominal, X_test_nominal, col, "onehot")
    ohe_train_frames.append(train_ohe_col)
    ohe_test_frames.append(test_ohe_col)

X_train_ohe = pd.concat(ohe_train_frames, axis=1) if ohe_train_frames else pd.DataFrame(index=X_train.index)
X_test_ohe = pd.concat(ohe_test_frames, axis=1) if ohe_test_frames else pd.DataFrame(index=X_test.index)

# Binary encoding pour les nominales de forte cardinalite.
bin_train_frames = []
bin_test_frames = []
for col in high_card_nominal_cols:
    train_bin_col, test_bin_col, _ = fit_transform_cat_column(X_train_nominal, X_test_nominal, col, "binary")
    bin_train_frames.append(train_bin_col)
    bin_test_frames.append(test_bin_col)

X_train_bin = pd.concat(bin_train_frames, axis=1) if bin_train_frames else pd.DataFrame(index=X_train.index)
X_test_bin = pd.concat(bin_test_frames, axis=1) if bin_test_frames else pd.DataFrame(index=X_test.index)

print("Shape ordinal train :", X_train_ord.shape)
print("Shape one-hot train :", X_train_ohe.shape)
print("Shape binary train  :", X_train_bin.shape)


In [ ]:
# Assemblage final et normalisation des noms de colonnes.
X_train_model = pd.concat([X_train_num, X_train_ord, X_train_ohe, X_train_bin], axis=1)
X_test_model = pd.concat([X_test_num, X_test_ord, X_test_ohe, X_test_bin], axis=1)
X_train_model = X_train_model.loc[:, ~X_train_model.columns.duplicated()].copy()
X_test_model = X_test_model.loc[:, X_train_model.columns].copy()

def sanitize_feature_names(columns):
    seen = {}
    output = []
    for col in columns:
        clean = str(col).strip().lower()
        clean = re.sub(r"[^0-9a-zA-Z]+", "_", clean)
        clean = re.sub(r"_+", "_", clean).strip("_") or "feature"
        base = clean
        idx = 1
        while clean in seen:
            idx += 1
            clean = f"{base}_{idx}"
        seen[clean] = True
        output.append(clean)
    return output

sanitized_feature_names = sanitize_feature_names(X_train_model.columns)
X_train_model.columns = sanitized_feature_names
X_test_model.columns = sanitized_feature_names
X_train_model = X_train_model.astype(np.float32)
X_test_model = X_test_model.astype(np.float32)

# Verifications de coherence avant export.
assert X_train_model.columns.tolist() == X_test_model.columns.tolist()
assert X_train_model.select_dtypes(include=["object", "string", "category"]).empty
assert X_test_model.select_dtypes(include=["object", "string", "category"]).empty
assert int(X_train_model.isna().sum().sum()) == 0
assert int(X_test_model.isna().sum().sum()) == 0

df_app_train_model_ready = pd.concat(
    [X_train[[id_col]].reset_index(drop=True), y_train.reset_index(drop=True), X_train_model.reset_index(drop=True)],
    axis=1,
)
df_app_test_model_ready = pd.concat(
    [X_test[[id_col]].reset_index(drop=True), X_test_model.reset_index(drop=True)],
    axis=1,
)

X_train_ready = df_app_train_model_ready.drop(columns=[id_col, target_col]).copy()
y_train_ready = df_app_train_model_ready[target_col].astype(int).copy()
X_test_ready = df_app_test_model_ready.drop(columns=[id_col]).copy()

print("Train model ready :", df_app_train_model_ready.shape)
print("Test model ready  :", df_app_test_model_ready.shape)
print("Nb features finales :", X_train_ready.shape[1])


In [ ]:
# Export CSV des jeux finaux et du plan de preprocessing.
output_dir = project_root / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

train_model_ready_csv = output_dir / "df_app_train_model_ready.csv"
test_model_ready_csv = output_dir / "df_app_test_model_ready.csv"
preprocessing_plan_csv = output_dir / "preprocessing_plan.csv"

df_app_train_model_ready.to_csv(train_model_ready_csv, index=False)
df_app_test_model_ready.to_csv(test_model_ready_csv, index=False)
preprocessing_plan.to_csv(preprocessing_plan_csv, index=False)

print(train_model_ready_csv)
print(test_model_ready_csv)
print(preprocessing_plan_csv)


In [ ]:
# Export PostgreSQL des tables finales preprocess ees.
def map_dtype_to_postgres(dtype):
    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"
    if pd.api.types.is_float_dtype(dtype):
        return "DOUBLE PRECISION"
    if pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "TIMESTAMP"
    return "TEXT"

def create_table_from_dataframe(conn, schema_name, table_name, df):
    column_defs = []
    for col_name, dtype in df.dtypes.items():
        column_defs.append(f'"{col_name}" {map_dtype_to_postgres(dtype)}')
    with conn.cursor() as cur:
        cur.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema_name}"')
        cur.execute(f'DROP TABLE IF EXISTS "{schema_name}"."{table_name}"')
        cur.execute(f'CREATE TABLE "{schema_name}"."{table_name}" ({", ".join(column_defs)})')
    conn.commit()

def copy_dataframe_to_table(conn, schema_name, table_name, df):
    buffer = StringIO()
    df.to_csv(buffer, index=False)
    buffer.seek(0)
    copy_sql = f'''
    COPY "{schema_name}"."{table_name}" FROM STDIN
    WITH (FORMAT CSV, HEADER TRUE, DELIMITER ',', QUOTE '"')
    '''
    with conn.cursor() as cur:
        cur.copy_expert(copy_sql, buffer)
    conn.commit()

create_table_from_dataframe(conn, "processed", "app_train_model_ready", df_app_train_model_ready)
copy_dataframe_to_table(conn, "processed", "app_train_model_ready", df_app_train_model_ready)
create_table_from_dataframe(conn, "processed", "app_test_model_ready", df_app_test_model_ready)
copy_dataframe_to_table(conn, "processed", "app_test_model_ready", df_app_test_model_ready)

final_check_query = """
SELECT 'processed.app_train_model_ready' AS table_name, COUNT(*) AS n_rows FROM processed.app_train_model_ready
UNION ALL
SELECT 'processed.app_test_model_ready' AS table_name, COUNT(*) AS n_rows FROM processed.app_test_model_ready
"""

display(pd.read_sql(final_check_query, conn))


## Resultat

A la fin de ce notebook, `X_train_ready`, `y_train_ready` et `X_test_ready` sont directement exploitables pour la suite de la modelisation.

Les jeux finalises sont :

- sans valeurs manquantes ;
- sans variables categorielles brutes ;
- alignes entre train et test ;
- disponibles a la fois sur disque et en base PostgreSQL.
